# 5 — Version Validation (optional, diagnostic)

Validates that the **same model version** appears at every layer of the inference pipeline. Use this after a deployment to confirm rollouts landed cleanly, or when investigating "is my endpoint serving the version I think it is?"

**Layers checked (top-to-bottom):**

| Layer | Source of truth |
|-------|----------------|
| MLflow Model Registry | `mlflow.search_model_versions(name='<MLFLOW_MODEL_NAME>')` |
| SageMaker Endpoint env | `describe_model` → `PrimaryContainer.Environment.MODEL_VERSION` |
| Inference response metadata | live `invoke_endpoint` → `response.metadata.model_version` |
| Athena `inference_responses` | logged `model_version` column (last 24 hours) |
| Athena `monitoring_responses` | logged `mlflow_run_id` column (most recent 10 drift runs) |

**Mismatch ⇒ something is wrong** — either the deploy didn't fully propagate, or two different model versions are silently being served, or version tagging in `inference.py`/`lambda_drift_monitor.py` is broken.

Run anytime — read-only. Sends exactly one test inference request.

[View on nbviewer](https://nbviewer.org/github/aws-samples/mlops-best-practices-on-aws/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/5_optional_version_validation.ipynb)

In [ ]:
import sys
import json
import boto3
import pandas as pd
from pathlib import Path

# Add project root to path
_project_root = Path.cwd().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

# Load environment
from dotenv import load_dotenv
load_dotenv(_project_root / '.env')

from src.config.config import (
    MLFLOW_TRACKING_URI,
    ATHENA_DATABASE,
)
from src.utils.mlflow_utils import setup_mlflow_tracking
from src.train_pipeline.athena.athena_client import AthenaClient
from mlflow.tracking import MlflowClient

print("✓ Imports successful")

## Configuration

In [ ]:
# Configuration
ENDPOINT_NAME = "fraud-detector-endpoint"  # Update with your endpoint name
MLFLOW_MODEL_NAME = "fraud-detection"

# Initialize clients
setup_mlflow_tracking(MLFLOW_TRACKING_URI)
mlflow_client = MlflowClient()
sagemaker_client = boto3.client('sagemaker')
runtime_client = boto3.client('sagemaker-runtime')
athena_client = AthenaClient()

print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Model: {MLFLOW_MODEL_NAME}")

## 1. Query MLflow Model Registry

In [ ]:
# Get all model versions from MLflow
model_versions = mlflow_client.search_model_versions(f"name='{MLFLOW_MODEL_NAME}'")

# Create summary DataFrame
mlflow_data = []
for mv in model_versions:
    mlflow_data.append({
        'version': int(mv.version),
        'stage': mv.current_stage,
        'run_id': mv.run_id[:8],
        'creation_time': mv.creation_timestamp,
        'status': mv.status
    })

mlflow_df = pd.DataFrame(mlflow_data).sort_values('version', ascending=False)

print("\n=" * 80)
print("MLflow Model Versions")
print("=" * 80)
print(mlflow_df.to_string(index=False))

if len(mlflow_df) > 0:
    latest_mlflow_version = mlflow_df['version'].max()
    print(f"\n✓ Latest MLflow version: {latest_mlflow_version}")
else:
    print("\n✗ No model versions found in MLflow")
    latest_mlflow_version = None

## 2. Query SageMaker Endpoint Configuration

In [ ]:
# Get endpoint details
try:
    endpoint = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_status = endpoint['EndpointStatus']
    endpoint_config_name = endpoint['EndpointConfigName']
    endpoint_arn = endpoint['EndpointArn']

    print()
    print("=" * 80)
    print("SageMaker Endpoint")
    print("=" * 80)
    print(f"Name:   {ENDPOINT_NAME}")
    print(f"Status: {endpoint_status}")
    print(f"Config: {endpoint_config_name}")
    print(f"ARN:    {endpoint_arn}")

    # Endpoint config → variant → model
    config = sagemaker_client.describe_endpoint_config(
        EndpointConfigName=endpoint_config_name
    )
    model_name = config['ProductionVariants'][0]['ModelName']
    print(f"Model:  {model_name}")

    # describe_model returns either PrimaryContainer (single-container model
    # built from image + ModelDataUrl) OR Containers[0] (model built from a
    # Model Registry ModelPackage). Handle both shapes.
    model_details = sagemaker_client.describe_model(ModelName=model_name)
    container = (
        model_details.get('PrimaryContainer')
        or (model_details.get('Containers') or [{}])[0]
    )
    env_vars = container.get('Environment', {})
    model_package_arn = container.get('ModelPackageName')  # only set for MP-based models

    sagemaker_version = env_vars.get('MODEL_VERSION', 'NOT SET')
    sagemaker_run_id = env_vars.get('MLFLOW_RUN_ID', 'NOT SET')

    print(f"\nEnvironment Variables:")
    print(f"  MODEL_VERSION: {sagemaker_version}")
    print(f"  MLFLOW_RUN_ID: {sagemaker_run_id}")
    if model_package_arn:
        print(f"  (ModelPackage: {model_package_arn})")

    if sagemaker_version != 'NOT SET':
        print(f"\n✓ SageMaker MODEL_VERSION: {sagemaker_version}")
    else:
        print("\n✗ MODEL_VERSION not set in endpoint environment")

except Exception as e:
    print(f"\n✗ Error querying SageMaker endpoint: {e}")
    sagemaker_version = None
    sagemaker_run_id = None


## 3. Test Inference and Extract Version from Response

In [ ]:
# Send a single test inference request to extract the response metadata.
#
# NOTE: the actual model trained from notebook 1 expects ~28 numeric features
# from the Kaggle credit-card-fraud dataset (Time, V1..V28, Amount, customer_age,
# etc.). The minimal payload below will fail validation in the endpoint — this
# is intentional. The block below catches the error and the rest of the
# notebook falls back to reading version info from Athena, which is what most
# users actually want here.
test_input = {
    "transaction_amount": 250.75,
    "customer_age": 42,
    "transaction_hour": 15,
    "is_international": 0,
    "transaction_count_1d": 8,
}

try:
    response = runtime_client.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(test_input)
    )
    result = json.loads(response['Body'].read().decode())
    print('=' * 80)
    print('Inference Response')
    print('=' * 80)
    print(json.dumps(result, indent=2))
    
    if 'metadata' in result:
        inference_version = result['metadata'].get('model_version', 'NOT FOUND')
        inference_run_id  = result['metadata'].get('mlflow_run_id', 'NOT FOUND')
        inference_endpoint = result['metadata'].get('endpoint_name', 'NOT FOUND')
        print(f'\n✓ Inference response includes metadata:')
        print(f'  model_version: {inference_version}')
        print(f'  mlflow_run_id: {inference_run_id}')
        print(f'  endpoint_name: {inference_endpoint}')
    else:
        print(f'\n✗ Response does not include metadata field')
        inference_version = None
        inference_run_id = None
except Exception as e:
    print(f'\n✗ Inference invocation failed: {e.__class__.__name__}')
    print(f'  (Expected if test payload schema does not match trained model;')
    print(f'   subsequent cells will fall back to Athena-logged metadata.)')
    inference_version = None
    inference_run_id = None


## 4. Query Athena for Logged Versions

In [ ]:
# Query Athena for recent inferences
try:
    query = f"""
        SELECT 
            model_version,
            mlflow_run_id,
            COUNT(*) as inference_count,
            MIN(request_timestamp) as first_seen,
            MAX(request_timestamp) as last_seen
        FROM {ATHENA_DATABASE}.inference_responses
        WHERE endpoint_name = '{ENDPOINT_NAME}'
          AND request_timestamp > CURRENT_TIMESTAMP - INTERVAL '24' HOUR
        GROUP BY model_version, mlflow_run_id
        ORDER BY last_seen DESC
    """
    
    athena_df = athena_client.execute_query(query)
    
    print("\n" + "=" * 80)
    print("Athena Logged Versions (Last 24 hours)")
    print("=" * 80)
    
    if not athena_df.empty:
        print(athena_df.to_string(index=False))
        athena_version = athena_df['model_version'].iloc[0]
        athena_run_id = athena_df['mlflow_run_id'].iloc[0]
        print(f"\n✓ Most recent Athena version: {athena_version}")
    else:
        print("No inference records found in last 24 hours")
        athena_version = None
        athena_run_id = None
        
except Exception as e:
    print(f"\n✗ Error querying Athena: {e}")
    athena_version = None
    athena_run_id = None

### Validation: Drift Monitoring Run IDs

Verify that each drift monitoring run has a unique MLflow run ID in the `monitoring_responses` table.

In [ ]:
# Query monitoring_responses for drift test runs
try:
    query = f"""
        SELECT
            monitoring_run_id,
            monitoring_timestamp,
            mlflow_run_id,
            data_drift_detected,
            model_drift_detected,
            drifted_columns_count,
            current_roc_auc,
            detection_engine
        FROM {ATHENA_DATABASE}.monitoring_responses
        ORDER BY monitoring_timestamp DESC
        LIMIT 10
    """
    
    athena_df = athena_client.execute_query(query)
    
    print("\n" + "=" * 80)
    print("Drift Monitoring Runs (Most Recent 10)")
    print("=" * 80)
    
    if not athena_df.empty:
        # Display table
        print(athena_df[['monitoring_timestamp', 'mlflow_run_id', 'data_drift_detected', 'model_drift_detected']].to_string(index=False))
        print(f"\n✓ Total drift monitoring runs: {len(athena_df)}")
        
        # Check for unique MLflow run IDs
        total_runs = len(athena_df)
        unique_runs = athena_df['mlflow_run_id'].nunique()
        null_runs = athena_df['mlflow_run_id'].isnull().sum()
        
        print(f"✓ Unique MLflow run IDs: {unique_runs}")
        
        if null_runs > 0:
            print(f"⚠️  NULL run IDs found: {null_runs} (Lambda failed to write mlflow_run_id — see docs/VERSION_MANAGEMENT.md)")
        
        if unique_runs == total_runs and null_runs == 0:
            print("✅ Each drift test has a unique MLflow run ID - tracking working correctly!")
        elif unique_runs < total_runs:
            print("⚠️  Some drift tests share MLflow run IDs (check for issues)")
    else:
        print("No drift monitoring runs found")
        print("\n💡 Run drift monitoring first:")
        print("   - Notebook 2_inference_monitoring.ipynb, Cell 62 (manual test)")
        print("   - Or wait for scheduled EventBridge trigger")
        
except Exception as e:
    print(f"\n✗ Error querying monitoring_responses: {e}")
    print("\n💡 Make sure drift monitoring has run at least once")

## 5. Version Consistency Validation

In [ ]:
# Version consistency validation. Compares the three sources of truth for
# "what model is serving traffic": the endpoint's configured MODEL_VERSION
# env var (sagemaker_version), the response body from a live invoke
# (inference_version — often None because inference.py doesn't attach
# metadata to the response body), and what Athena logs report for actual
# recent predictions (athena_version — the ground truth once traffic has
# flowed).
validation_results = []

print("\n" + "=" * 80)
print("VERSION CONSISTENCY VALIDATION")
print("=" * 80)

# Show what data we have
print(f"\n  Sources available:")
print(f"    sagemaker_version:  {sagemaker_version or '(unavailable)'}")
print(f"    inference_version:  {inference_version or '(unavailable — inference.py returns no metadata)'}")
print(f"    athena_version:     {athena_version or '(unavailable — no logs in last 24h)'}")
print(f"    sagemaker_run_id:   {sagemaker_run_id or '(unavailable)'}")
print(f"    inference_run_id:   {inference_run_id or '(unavailable)'}")
print(f"    athena_run_id:      {athena_run_id or '(unavailable)'}")

def add_check(name, expected, actual, level='FAIL'):
    """level: 'FAIL' for hard mismatch, 'WARNING' for softer (e.g. run-id)."""
    if expected == actual:
        status = 'PASS';  glyph = '✓'
    else:
        status = level;   glyph = '⚠' if level == 'WARNING' else '✗'
    validation_results.append({'Check': name, 'Expected': expected, 'Actual': actual, 'Status': status})
    print(f"\n{glyph} {name}: expected={expected!r} actual={actual!r} → {status}")

# Check 1: SageMaker env vs Athena logs (the useful one — does the endpoint's
# advertised version match what's actually landing in inference logs?)
if sagemaker_version and athena_version:
    add_check('SageMaker env vs Athena logs (MODEL_VERSION)', sagemaker_version, athena_version)
else:
    print("\n  Skipping SageMaker↔Athena version check (missing one side).")

# Check 2: SageMaker env vs Inference response (only meaningful if inference.py
# was patched to attach metadata — currently a no-op).
if sagemaker_version and inference_version:
    add_check('SageMaker env vs Inference response (MODEL_VERSION)', sagemaker_version, inference_version)

# Check 3: Inference response vs Athena logs (also no-op currently)
if inference_version and athena_version:
    add_check('Inference response vs Athena logs (MODEL_VERSION)', inference_version, athena_version)

# Check 4: MLflow run id — sagemaker vs athena. Warning-level: mismatch
# is common when a model is deployed from Model Registry (sagemaker_run_id
# is a synthetic label like 'model-registry-v1') vs an MLflow-linked run
# (athena_run_id is a real MLflow run id).
if sagemaker_run_id and athena_run_id:
    add_check('MLflow run id: SageMaker env vs Athena logs',
              sagemaker_run_id, athena_run_id, level='WARNING')

# Summary
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

if not validation_results:
    print("No checks ran — need at least SageMaker env vars AND Athena inference logs.")
    print("Fix: run inference traffic against the endpoint, wait ~30s for Athena logs, retry.")
else:
    validation_df = pd.DataFrame(validation_results)
    print(validation_df.to_string(index=False))
    counts = validation_df['Status'].value_counts().to_dict()
    fail = counts.get('FAIL', 0)
    warn = counts.get('WARNING', 0)
    passed = counts.get('PASS', 0)
    print(f"\nResults: {passed} passed, {warn} warnings, {fail} failed")
    if fail == 0:
        print("\n✓ All version consistency checks PASSED!")
    else:
        print(f"\n✗ {fail} version consistency check(s) FAILED!")


## 6. Version Distribution Analysis

In [ ]:
# Get version distribution
try:
    dist = athena_client.get_version_distribution(
        endpoint_name=ENDPOINT_NAME,
        hours=24
    )
    
    print("\n" + "=" * 80)
    print("Version Distribution (Last 24 hours)")
    print("=" * 80)
    
    if not dist.empty:
        print(dist.to_string(index=False))
        
        # Check for version drift
        if len(dist) > 1:
            print(f"\n⚠️  WARNING: Multiple versions detected!")
            print(f"   This may indicate a rollback or deployment issue.")
            print(f"   Versions: {', '.join(dist['model_version'].tolist())}")
    else:
        print("No inferences found in last 24 hours")
        
except Exception as e:
    print(f"Error getting version distribution: {e}")

## 7. Version Drift Detection

In [ ]:
# Detect version drift
try:
    drift_result = athena_client.detect_version_drift(
        endpoint_name=ENDPOINT_NAME,
        hours=1
    )
    
    print("\n" + "=" * 80)
    print("Version Drift Detection (Last 1 hour)")
    print("=" * 80)
    print(f"Has Drift: {drift_result['has_drift']}")
    print(f"Version Count: {drift_result['version_count']}")
    print(f"Versions: {drift_result['versions']}")
    print(f"\n{drift_result['drift_message']}")
    
except Exception as e:
    print(f"Error detecting version drift: {e}")

## 8. Version Performance Comparison

In [ ]:
# Compare performance across versions
try:
    perf = athena_client.get_version_performance_comparison(
        endpoint_name=ENDPOINT_NAME,
        days=7
    )
    
    print("\n" + "=" * 80)
    print("Version Performance Comparison (Last 7 days)")
    print("=" * 80)
    
    if not perf.empty:
        # Select key columns
        display_cols = [
            'model_version',
            'total_predictions',
            'fraud_rate',
            'avg_latency_ms',
            'high_confidence_rate'
        ]
        print(perf[display_cols].to_string(index=False))
    else:
        print("No data for performance comparison")
        
except Exception as e:
    print(f"Error getting performance comparison: {e}")

## Summary

This notebook diagnoses **version traceability across the inference pipeline**:
1. MLflow Model Registry — source of truth for trained versions
2. SageMaker endpoint env — what the deployed container thinks it's running
3. Inference response metadata — what the endpoint reports per request
4. Athena `inference_responses` — what's been logged historically
5. Athena `monitoring_responses` — what each drift run pegged itself against

If any of the four checks **FAIL**, the most common causes:
- Endpoint redeployed with new model but old env vars cached → re-run notebook 2 cell 8
- `inference.py` doesn't read `MODEL_VERSION` from env → check `src/train_pipeline/pipeline_steps/inference.py`
- Lambda logger not running → check `InferenceLoggerFunction` in CFN

For background on version-tagging design, see `docs/VERSION_MANAGEMENT.md`.